## Elementos técnicos del formato CSV

|Nombre|Carácter|Parámetro sep|Uso común|
|:----|:----|:----|:----|
|Coma|,|sep=','|El estándar global para archivos .csv.|
|Punto y coma|;|sep=';'|Muy común en archivos generados por Excel en regiones de habla hispana o europea (donde la coma es el separador decimal).|
|Tabulación|\t|sep='\t'|Formato .tsv (Tab-Separated Values). Muy usado en bioinformática y logs.|
|Pipe (Barra)|\||`sep='|'`|
|Espacio| |sep='\s+'|Archivos de texto plano de ancho fijo o reportes de consola.|
|Dos puntos|:|sep=':'|Archivos de configuración de sistemas Linux (ej. /etc/passwd).|


## Cargar un Dataset con Pandas

Nota: La funcion **df.info(memory_usage="deep")** nos ayuda a saber la memoria consumida para cargar el dataframe en memoria.

In [ ]:
import pandas as pd

df = pd.read_csv("twitch_es_mar_26.csv", sep="\t")
df.info(memory_usage='deep')


## Cargar solo cabecera del CSV para saber qué columnas nos interesan

El argumento **nrows=0** nos carga únicamente la cabecera del csv.

In [ ]:
import pandas as pd

# Carga solo los metadatos (columnas)
df = pd.read_csv('twitch_es_mar_26.csv', sep="\t", nrows=0)
df.info(memory_usage='deep')
# Ver la lista de columnas
print(df.columns.tolist())

## Cargar Solo ciertas columnas

In [ ]:
import pandas as pd

df = pd.read_csv('twitch_es_mar_25.csv', sep="\t", usecols=['captured_at',"position"])
df.info(memory_usage='deep')


## Guardar el Dataset con las columnas que nos interesan

In [ ]:
import pandas as pd

df = pd.read_csv('twitch_es_mar_25.csv', sep="\t", usecols=['captured_at',"position"])
df.info(memory_usage='deep')
df.to_csv("dataset_limpio.csv", sep="\t", index=False)

## Calcular promedios y sumas Agregadas por fecha de captura

In [ ]:
import pandas as pd

df = pd.read_csv('twitch_es_mar_26.csv', sep="\t", usecols=['captured_at',"streamer_id", "viewer_count"])
df.info(memory_usage='deep')

df_resumen = df.groupby('captured_at').agg({
    'viewer_count': [
        'sum', 
        'mean',
        'median',
        'std',
        'max'
        ],
    'streamer_id': 'nunique'
})

df_resumen.columns = [
    'total_viewers', 
    'avg_viewers', 
    'median_viewers',
    'std_viewers',
    'max_viewers',
    'unique_streamers']
df_resumen = df_resumen.reset_index()

# Redondear decimales a 2
df_resumen["avg_viewers"] =  df_resumen["avg_viewers"].round(2)
df_resumen["median_viewers"] =  df_resumen["median_viewers"].round(2)
df_resumen["std_viewers"] =  df_resumen["std_viewers"].round(2)

df_resumen.to_csv('resumen_metricas_2026.csv', sep="\t", index=False)
df_resumen.info(memory_usage='deep')

print("¡Archivo generado con éxito!")
print(df_resumen.head())

## Concatenar dos datasets

In [12]:
# EN VERTICAL

import pandas as pd

df1 = pd.read_csv("resumen_metricas_2025.csv", sep="\t")
df2 = pd.read_csv("resumen_metricas_2026.csv", sep="\t")

df_concatenada = pd.concat([df1, df2], ignore_index=True)
df_concatenada.to_csv("df_concatenada_final_25_26.csv", sep="\t", index=False, decimal=",")


## Estadísticas más profundas Quartiles

In [ ]:
import pandas as pd

df = pd.read_csv('twitch_es_mar_26.csv', sep="\t", usecols=['captured_at',"streamer_id", "viewer_count"])
df.info(memory_usage='deep')

df_resumen = df.groupby('captured_at').agg({
    'viewer_count': [
        ('q1', lambda x: x.quantile(0.25)), 
        ('q2', lambda x: x.quantile(0.50)), 
        ('q3', lambda x: x.quantile(0.75)),
        ],
    'streamer_id': 'nunique'
})

df_resumen.columns = [
    'q25',
    'q50',
    'q75',
    'unique_streamers']
df_resumen = df_resumen.reset_index()

df_resumen.to_csv('resumen_quartiles.csv', sep="\t", index=False)
df_resumen.info(memory_usage='deep')

print("¡Archivo generado con éxito!")
print(df_resumen.head())

# Carga por CHUNKS

## Filtrar solo ciertas posiciones del Ranking en cada captura

Cuando un dataset supera la memoria ram disponible, debemos cargarlo y procesarlo por secciones. Cada una de estas secciones se llama "Chunk". 

En la lectura del CSV, se introduce el argumento *chunksize=chunk_size*. Donde chunk_size es el número de filas que vamos a cargar en cada iteración. 

In [ ]:
import pandas as pd

chunk_size = 100000  # Leer de 100k en 100k filas
dataset_salida = "nombre_dataset_salida.csv"

lista_fragmentos = []

for chunk in pd.read_csv("twitch_es_mar_25.csv", sep="\t", chunksize=chunk_size):
    mask = (chunk["position"] >= 0) & (chunk["position"] <= 100)
    chunk_filtrado = chunk[mask]
    if not chunk_filtrado.empty:
        lista_fragmentos.append(chunk_filtrado)

# Unimos todos los trozos filtrados en un solo DataFrame final
df_final = pd.concat(lista_fragmentos, ignore_index=True)
df_final.to_csv(f"{dataset_salida}", sep="\t", quotechar='"', index=False)
print("ok!")

# CONTROL DE MEMORIA CONSUMIDA en DF_FINAL
df_final.info(memory_usage='deep')

**Pregunta**: ¿Existe algún limite de tamaño para los datasets si se cargan por Chunks? 

**Pregunta**: ¿Qué pasa si la "lista_fragmentos" supera nuestra capacidad de RAM?